Notebook 04 - Baselines, Random Forest and XGBoost

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import os
import glob
import json
import joblib
import time

from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report,
)
from xgboost import XGBClassifier

import warnings
warnings.filterwarnings("ignore")


Mounted at /content/drive


In [ ]:
PROJECT_PATH = "/content/drive/MyDrive/AI_Trading_Dissertation"
FINAL_PROJECT_PATH = PROJECT_PATH + "/Final_Project"

SPLITS_PATH = FINAL_PROJECT_PATH + "/02_Data/Splits"
MODELS_PATH = FINAL_PROJECT_PATH + "/03_Models"
RESULTS_PATH = FINAL_PROJECT_PATH + "/04_Results"
PREDICTIONS_PATH = RESULTS_PATH + "/Predictions"

for sub in ["Baseline", "RandomForest", "XGBoost"]:
    os.makedirs(os.path.join(MODELS_PATH, sub), exist_ok=True)
os.makedirs(PREDICTIONS_PATH, exist_ok=True)

# Fixing label order everywhere - so every confusion matrix in this project has the same row/column meaning and can be compared directly across models and stocks.
LABELS = ["Buy", "Hold", "Sell"]

RANDOM_STATE = 42  # fixed so results are reproducible

print("Reading splits from:", SPLITS_PATH)


Reading splits from: /content/drive/MyDrive/AI_Trading_Dissertation/Final_Project/02_Data/Splits


In [ ]:
train_files = sorted(glob.glob(os.path.join(SPLITS_PATH, "*_train.parquet")))
stock_names = [os.path.basename(f).replace("_train.parquet", "") for f in train_files]

print("Stocks found:", len(stock_names))
print(stock_names)

# Recovering the feature list
_sample = pd.read_parquet(train_files[0])
EXCLUDE_FROM_FEATURES = {
    "datetime", "open", "high", "low", "close", "volume", "tick_count",
    "future_return", "target_threshold", "target_signal",
}
FEATURE_COLUMNS = [c for c in _sample.columns if c not in EXCLUDE_FROM_FEATURES]

print(f"\n{len(FEATURE_COLUMNS)} features:", FEATURE_COLUMNS)


Stocks found: 10
['Eicher_Motors', 'HDFCB', 'Hindustan_Unilever', 'ICICIBC', 'INFOSYS', 'ITC', 'Mahindra_and_Mahindra', 'ONGC', 'Reliance', 'TCS']

22 features: ['return_1', 'log_return_1', 'sma_5', 'ema_5', 'sma_10', 'ema_10', 'sma_20', 'ema_20', 'sma_50', 'ema_50', 'rsi_14', 'macd_line', 'macd_signal', 'macd_hist', 'bb_mid', 'bb_upper', 'bb_lower', 'bb_percent_b', 'bb_bandwidth', 'volatility_20', 'volume_avg_20', 'volume_relative']


In [ ]:
def momentum_baseline_predict(df):
    # If the last bar was up -> guess Buy. Down -> Sell. Exactly flat -> Hold.
    # Uses return_1, which is lookahead-safe.
    return np.where(df["return_1"] > 0, "Buy",
           np.where(df["return_1"] < 0, "Sell", "Hold"))


The Evaluation Function

In [ ]:
def evaluate_predictions(y_true, y_pred, stock, model_name, train_seconds=None):
    acc = accuracy_score(y_true, y_pred)

    # macro = average the metric across the 3 classes equally, so a model can't score well just by nailing the biggest class and ignoring the others.
    precision_macro = precision_score(y_true, y_pred, average="macro", labels=LABELS, zero_division=0)
    recall_macro = recall_score(y_true, y_pred, average="macro", labels=LABELS, zero_division=0)
    f1_macro = f1_score(y_true, y_pred, average="macro", labels=LABELS, zero_division=0)

    # weighted = same but weighted by how common each class is - closer to "overall" feel.
    f1_weighted = f1_score(y_true, y_pred, average="weighted", labels=LABELS, zero_division=0)

    # Traded accuracy: only the bars where the model actually said Buy or Sell.
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    traded_mask = np.isin(y_pred, ["Buy", "Sell"])
    n_trades = int(traded_mask.sum())
    traded_acc = float(accuracy_score(y_true[traded_mask], y_pred[traded_mask])) if n_trades > 0 else np.nan

    row = {
        "Stock": stock,
        "Model": model_name,
        "Accuracy": round(acc, 4),
        "Precision_Macro": round(precision_macro, 4),
        "Recall_Macro": round(recall_macro, 4),
        "F1_Macro": round(f1_macro, 4),
        "F1_Weighted": round(f1_weighted, 4),
        "Traded_Accuracy": round(traded_acc, 4) if not np.isnan(traded_acc) else np.nan,
        "N_Trade_Signals": n_trades,
        "Trade_Signal_Rate": round(n_trades / len(y_pred), 4),
        "Train_Seconds": round(train_seconds, 1) if train_seconds is not None else None,
    }

    cm = confusion_matrix(y_true, y_pred, labels=LABELS)
    return row, cm


Training and Evaluating Everything

In [ ]:
all_results = []
all_confusion = {}
all_feature_importances = []

overall_start = time.time()

for i, stock in enumerate(stock_names, 1):
    print(f"\n{'=' * 70}")
    print(f"[{i}/{len(stock_names)}] {stock}")
    print("=" * 70)

    train_df = pd.read_parquet(os.path.join(SPLITS_PATH, f"{stock}_train.parquet"))
    test_df = pd.read_parquet(os.path.join(SPLITS_PATH, f"{stock}_test.parquet"))

    X_train = train_df[FEATURE_COLUMNS]
    y_train = train_df["target_signal"]
    X_test = test_df[FEATURE_COLUMNS]
    y_test = test_df["target_signal"]

    print(f"  train {X_train.shape}, test {X_test.shape}")


    preds_out = test_df[["datetime", "close", "future_return", "target_signal"]].copy()

    # Baseline 1: stratified random
    t0 = time.time()
    dummy_random = DummyClassifier(strategy="stratified", random_state=RANDOM_STATE)
    dummy_random.fit(X_train, y_train)
    pred = dummy_random.predict(X_test)
    row, cm = evaluate_predictions(y_test, pred, stock, "Baseline_Random", time.time() - t0)
    all_results.append(row); all_confusion[(stock, "Baseline_Random")] = cm
    preds_out["pred_Baseline_Random"] = pred
    print(f"  Baseline_Random      acc={row['Accuracy']:.4f}  F1={row['F1_Macro']:.4f}")

    #  Baseline 2: majority class
    t0 = time.time()
    dummy_major = DummyClassifier(strategy="most_frequent")
    dummy_major.fit(X_train, y_train)
    pred = dummy_major.predict(X_test)
    row, cm = evaluate_predictions(y_test, pred, stock, "Baseline_Majority", time.time() - t0)
    all_results.append(row); all_confusion[(stock, "Baseline_Majority")] = cm
    preds_out["pred_Baseline_Majority"] = pred
    print(f"  Baseline_Majority    acc={row['Accuracy']:.4f}  F1={row['F1_Macro']:.4f}")

    #  Baseline 3: momentum rule
    t0 = time.time()
    pred = momentum_baseline_predict(test_df)
    row, cm = evaluate_predictions(y_test, pred, stock, "Baseline_Momentum", time.time() - t0)
    all_results.append(row); all_confusion[(stock, "Baseline_Momentum")] = cm
    preds_out["pred_Baseline_Momentum"] = pred
    print(f"  Baseline_Momentum    acc={row['Accuracy']:.4f}  F1={row['F1_Macro']:.4f}")

    #  Random Forest
    t0 = time.time()
    rf = RandomForestClassifier(
        n_estimators=200,
        max_depth=12,          # capped: unlimited depth memorises noise on financial data
        min_samples_leaf=50,   # each leaf must cover >=50 bars - guards against overfitting
        n_jobs=-1,
        random_state=RANDOM_STATE,
        class_weight="balanced_subsample",
    )
    rf.fit(X_train, y_train)
    rf_seconds = time.time() - t0
    pred = rf.predict(X_test)
    row, cm = evaluate_predictions(y_test, pred, stock, "RandomForest", rf_seconds)
    all_results.append(row); all_confusion[(stock, "RandomForest")] = cm
    preds_out["pred_RandomForest"] = pred
    print(f"  RandomForest         acc={row['Accuracy']:.4f}  F1={row['F1_Macro']:.4f}  ({rf_seconds:.0f}s)")

    joblib.dump(rf, os.path.join(MODELS_PATH, "RandomForest", f"{stock}_rf.joblib"))

    for feat, imp in zip(FEATURE_COLUMNS, rf.feature_importances_):
        all_feature_importances.append({"Stock": stock, "Model": "RandomForest", "Feature": feat, "Importance": imp})

    # XGBoost
    # XGBoost needs numeric labels, so mapping Buy/Hold/Sell -> 0/1/2 and mapping back after.
    label_to_int = {label: idx for idx, label in enumerate(LABELS)}
    int_to_label = {idx: label for label, idx in label_to_int.items()}

    t0 = time.time()
    xgb = XGBClassifier(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        tree_method="hist",
        eval_metric="mlogloss",
    )
    xgb.fit(X_train, y_train.map(label_to_int))
    xgb_seconds = time.time() - t0
    pred = pd.Series(xgb.predict(X_test)).map(int_to_label).values
    row, cm = evaluate_predictions(y_test, pred, stock, "XGBoost", xgb_seconds)
    all_results.append(row); all_confusion[(stock, "XGBoost")] = cm
    preds_out["pred_XGBoost"] = pred
    print(f"  XGBoost              acc={row['Accuracy']:.4f}  F1={row['F1_Macro']:.4f}  ({xgb_seconds:.0f}s)")

    xgb.save_model(os.path.join(MODELS_PATH, "XGBoost", f"{stock}_xgb.json"))

    for feat, imp in zip(FEATURE_COLUMNS, xgb.feature_importances_):
        all_feature_importances.append({"Stock": stock, "Model": "XGBoost", "Feature": feat, "Importance": imp})

    # Saving stock's predictions for backtesting
    preds_out.to_parquet(os.path.join(PREDICTIONS_PATH, f"{stock}_predictions.parquet"), index=False)

print(f"\n\nAll done in {(time.time() - overall_start) / 60:.1f} minutes.")



[1/10] Eicher_Motors
  train (57953, 22), test (14489, 22)
  Baseline_Random      acc=0.3396  F1=0.3357
  Baseline_Majority    acc=0.3580  F1=0.1757
  Baseline_Momentum    acc=0.3335  F1=0.2750
  RandomForest         acc=0.3940  F1=0.3911  (61s)
  XGBoost              acc=0.3855  F1=0.3695  (13s)

[2/10] HDFCB
  train (58344, 22), test (14587, 22)
  Baseline_Random      acc=0.3424  F1=0.3379
  Baseline_Majority    acc=0.3529  F1=0.1739
  Baseline_Momentum    acc=0.3394  F1=0.2794
  RandomForest         acc=0.3925  F1=0.3650  (71s)
  XGBoost              acc=0.3851  F1=0.3620  (10s)

[3/10] Hindustan_Unilever
  train (58308, 22), test (14578, 22)
  Baseline_Random      acc=0.3402  F1=0.3350
  Baseline_Majority    acc=0.3616  F1=0.1770
  Baseline_Momentum    acc=0.3400  F1=0.2802
  RandomForest         acc=0.3757  F1=0.3739  (59s)
  XGBoost              acc=0.3827  F1=0.3798  (12s)

[4/10] ICICIBC
  train (58424, 22), test (14607, 22)
  Baseline_Random      acc=0.3382  F1=0.3316
  Basel

The Results Table

In [ ]:
results_df = pd.DataFrame(all_results)
results_df.to_csv(os.path.join(RESULTS_PATH, "Model_Metrics_AllStocks.csv"), index=False)

print("Saved full results. Rows:", len(results_df))
results_df


Saved full results. Rows: 50


,Stock,Model,Accuracy,Precision_Macro,Recall_Macro,F1_Macro,F1_Weighted,Traded_Accuracy,N_Trade_Signals,Trade_Signal_Rate,Train_Seconds
0,Eicher_Motors,Baseline_Random,0.3396,0.3357,0.3357,0.3357,0.3399,0.3646,10314,0.7119,0.0
1,Eicher_Motors,Baseline_Majority,0.3580,0.1193,0.3333,0.1757,0.1888,0.3580,14489,1.0000,0.0
2,Eicher_Motors,Baseline_Momentum,0.3335,0.2822,0.3110,0.2750,0.2931,0.3416,13828,0.9544,0.0
3,Eicher_Motors,RandomForest,0.3940,0.3933,0.3959,0.3911,0.3904,0.3999,9956,0.6871,60.8
4,Eicher_Motors,XGBoost,0.3855,0.3985,0.3731,0.3695,0.3759,0.3769,12514,0.8637,13.1
5,HDFCB,Baseline_Random,0.3424,0.3380,0.3382,0.3379,0.3420,0.3632,10590,0.7260,0.0
6,HDFCB,Baseline_Majority,0.3529,0.1176,0.3333,0.1739,0.1841,0.3529,14587,1.0000,0.0
7,HDFCB,Baseline_Momentum,0.3394,0.2786,0.3160,0.2794,0.2985,0.3503,13849,0.9494,0.0
8,HDFCB,RandomForest,0.3925,0.3980,0.3982,0.3650,0.3608,0.3932,9297,0.6373,70.9
9,HDFCB,XGBoost,0.3851,0.3942,0.3721,0.3620,0.3676,0.3788,12246,0.8395,10.0


Average performance per model

In [ ]:
summary = (
    results_df
    .groupby("Model")[["Accuracy", "Precision_Macro", "Recall_Macro", "F1_Macro", "Traded_Accuracy", "Trade_Signal_Rate"]]
    .mean()
    .round(4)
    .sort_values("F1_Macro", ascending=False)
)

summary.to_csv(os.path.join(RESULTS_PATH, "Model_Metrics_Summary.csv"))
summary


,Accuracy,Precision_Macro,Recall_Macro,F1_Macro,Traded_Accuracy,Trade_Signal_Rate
Model,,,,,,
RandomForest,0.3938,0.3956,0.3983,0.3841,0.3985,0.6577
XGBoost,0.3892,0.3978,0.3786,0.3732,0.3816,0.8278
Baseline_Random,0.3390,0.3346,0.3346,0.3345,0.3619,0.7203
Baseline_Momentum,0.3391,0.2841,0.3157,0.2815,0.3496,0.9449
Baseline_Majority,0.3604,0.1201,0.3333,0.1766,0.3604,1.0000


In [ ]:
pivot_f1 = results_df.pivot(index="Stock", columns="Model", values="F1_Macro")
pivot_f1 = pivot_f1[["Baseline_Random", "Baseline_Majority", "Baseline_Momentum", "RandomForest", "XGBoost"]]
pivot_f1.to_csv(os.path.join(RESULTS_PATH, "F1_By_Stock_And_Model.csv"))
pivot_f1.round(4)


Model,Baseline_Random,Baseline_Majority,Baseline_Momentum,RandomForest,XGBoost
Stock,,,,,
Eicher_Motors,0.3357,0.1757,0.2750,0.3911,0.3695
HDFCB,0.3379,0.1739,0.2794,0.3650,0.3620
Hindustan_Unilever,0.3350,0.1770,0.2802,0.3739,0.3798
ICICIBC,0.3316,0.1776,0.2847,0.3683,0.3613
INFOSYS,0.3397,0.1700,0.2755,0.3849,0.4034
ITC,0.3387,0.1812,0.2906,0.3977,0.3764
Mahindra_and_Mahindra,0.3394,0.1750,0.2801,0.3740,0.3723
ONGC,0.3299,0.1810,0.2861,0.3913,0.3707
Reliance,0.3288,0.1837,0.2862,0.3998,0.3611


In [ ]:
for model_name in ["RandomForest", "XGBoost"]:
    print(f"\n{'#' * 70}")
    print(f"# {model_name}")
    print("#" * 70)
    for stock in stock_names:
        cm = all_confusion[(stock, model_name)]
        cm_df = pd.DataFrame(cm, index=[f"true_{l}" for l in LABELS], columns=[f"pred_{l}" for l in LABELS])
        print(f"\n{stock}:")
        print(cm_df)



######################################################################
# RandomForest
######################################################################

Eicher_Motors:
           pred_Buy  pred_Hold  pred_Sell
true_Buy       2436       1435       1365
true_Hold      1366       1727        973
true_Sell      2271       1371       1545

HDFCB:
           pred_Buy  pred_Hold  pred_Sell
true_Buy        652       1539       2957
true_Hold       254       2070       1760
true_Sell       670       1681       3004

Hindustan_Unilever:
           pred_Buy  pred_Hold  pred_Sell
true_Buy       1776       1937       1558
true_Hold       993       2189        802
true_Sell      1678       2133       1512

ICICIBC:
           pred_Buy  pred_Hold  pred_Sell
true_Buy        884       1755       2667
true_Hold       468       2018       1461
true_Sell       839       1801       2714

INFOSYS:
           pred_Buy  pred_Hold  pred_Sell
true_Buy       1004       1790       2205
true_Hold       586  

In [ ]:
cm_rows = []
for (stock, model_name), cm in all_confusion.items():
    for i, true_label in enumerate(LABELS):
        for j, pred_label in enumerate(LABELS):
            cm_rows.append({
                "Stock": stock, "Model": model_name,
                "True": true_label, "Predicted": pred_label, "Count": int(cm[i, j]),
            })

pd.DataFrame(cm_rows).to_csv(os.path.join(RESULTS_PATH, "Confusion_Matrices_All.csv"), index=False)
print("Confusion matrices saved.")


Confusion matrices saved.


In [ ]:
fi_df = pd.DataFrame(all_feature_importances)

top_features = (
    fi_df.groupby(["Model", "Feature"])["Importance"]
    .mean()
    .reset_index()
    .sort_values(["Model", "Importance"], ascending=[True, False])
)

fi_df.to_csv(os.path.join(RESULTS_PATH, "Feature_Importances_AllStocks.csv"), index=False)
top_features.to_csv(os.path.join(RESULTS_PATH, "Feature_Importances_Averaged.csv"), index=False)

for model_name in ["RandomForest", "XGBoost"]:
    print(f"\nTop 10 features — {model_name}:")
    print(top_features[top_features["Model"] == model_name].head(10).to_string(index=False))



Top 10 features — RandomForest:
       Model         Feature  Importance
RandomForest volume_relative    0.198362
RandomForest   volatility_20    0.164053
RandomForest    bb_bandwidth    0.068469
RandomForest    bb_percent_b    0.050317
RandomForest   volume_avg_20    0.045943
RandomForest     macd_signal    0.037795
RandomForest          rsi_14    0.037240
RandomForest       macd_line    0.037183
RandomForest       macd_hist    0.035850
RandomForest    log_return_1    0.034060

Top 10 features — XGBoost:
  Model         Feature  Importance
XGBoost volume_relative    0.080207
XGBoost   volatility_20    0.063712
XGBoost          bb_mid    0.050549
XGBoost          ema_50    0.049665
XGBoost          ema_10    0.049324
XGBoost          ema_20    0.049246
XGBoost        bb_lower    0.047851
XGBoost        bb_upper    0.047465
XGBoost          sma_50    0.047317
XGBoost          sma_20    0.046878
